# 08 | The economics model: two cohorts, channel CAC, break-even and gates

**Author: Chanakya**

The single source of truth for every economic figure in the recommendation. Two mutually exclusive cohorts are each measured against their own holdout. **Acquisition** buys new ATP pass buyers and is counted on an incremental basis. **Upgrade** moves existing ATP pass buyers to a season pass, and only upgrades above the control rate are credited, with the credit charged to every treated upgrader. The model lives in `models/atp_economics.py`; every input, with its evidence tag, is in `models/model_inputs.json`. Rights fees are excluded: this is an incremental campaign P&L, not a claim about total rights ROI.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='08_economics_model'
shared.ACTIVE_SOURCES=[]

In [ ]:
sys.path.insert(0, str(ROOT/'models'))
import atp_economics as m
res = m.run()
rows=[]
for section,items in m.INPUTS.items():
 if section.startswith('_'):continue
 for key,x in items.items():
  if key.startswith('_'):continue
  rows.append(dict(section=section,input=key,value=json.dumps(x['value']) if isinstance(x['value'],(list,dict)) else x['value'],tag=x.get('tag',''),source=x.get('source',''),note=x.get('note','')))
inputs=pd.DataFrame(rows);display(table(inputs,'08_model_inputs'))
print('Evidence tags:',inputs.tag.value_counts().to_dict())

## 1. Unit economics
Contribution = price / 1.18 GST, less a 2% gateway fee plus GST on the fee, less INR 30 variable cost. The upgrade credit costs less than its INR 44.50 face value because GST and the gateway fee fall with the price.

In [ ]:
u=pd.DataFrame([dict(item=k,value=v) for k,v in res['unit_economics'].items()]);display(table(u,'08_unit_economics'))

## 2. Channel economics and the INR 3.04 Cr conditional envelope
Channels that clear the brief’s INR 150–200 target on attributed CAC keep their envelope. Paid media, at INR 333, is capped to a INR 25 lakh external-audience test. The freed money is not moved into owned messaging, because owned reach is finite and its incremental CAC rises with depth. It becomes a **performance reserve**, released only to a channel whose measured marginal incremental CAC is at or below INR 200. Test-ceiling channels have no public cost basis; their CAC is a purchasing rule, not an estimate.

In [ ]:
ch=pd.DataFrame(res['channels']);display(table(ch,'08_channel_cac_and_net'))
env=pd.DataFrame([dict(item=k,value=v) for k,v in res['envelope'].items() if not isinstance(v,dict)]);display(table(env,'08_envelope_summary'))
gates=pd.DataFrame(res['gate_allocation']);display(table(gates,'08_gate_allocation'))
fig_,ax=plt.subplots(figsize=(10,4.2))
order=ch.sort_values('net_per_payer_24m_at_100')
ax.barh(order.label,order.net_per_payer_24m_at_100,color=[COLORS[3] if x< -1 else COLORS[1] for x in order.net_per_payer_24m_at_100],label='All attributed payers incremental')
ax.scatter(order.net_per_payer_24m_at_87_5,order.label,color='black',zorder=3,label='87.5% incremental')
ax.axvline(0,color='black',lw=.8);ax.set_xlabel('24-month contribution minus attributed CAC, INR per payer, before any upgrade')
ax.set_title('No acquisition channel pays back on pass purchases alone. Owned comes closest');ax.legend(loc='lower right',fontsize=8)
fig('08_channel_net_per_payer','Contribution per acquired payer over 24 months = 1.625 purchases x INR 43.32 x (1 + 40% year-two return) = INR 98.56. Scenario, not observed.')

## 3. Personas and capacity
In-app channels (owned, native) reach portfolio fans already on FanCode and are split in proportion to the base sport pools. External channels (contests, publisher, paid) reach the Slam Tourist. Pools can overlap and are a capacity check, not a forecast.

In [ ]:
pp=pd.DataFrame([dict(persona=k,attributed_payers=v,share=res['personas']['payer_share'][k]) for k,v in res['personas']['payers'].items()]);display(table(pp,'08_persona_payers'))
pools=pd.DataFrame([dict(scenario=k,**{s+'_m':x for s,x in d.items()}) for k,d in res['personas']['pools_m'].items()]);pools['payer_target_share_of_pools']=[res['personas']['capacity_share_of_pools'][k] for k in pools.scenario];display(table(pools,'08_sport_pools_capacity'))

## 4. The upgrade cohort and the 24-month P&L
Eligible pass buyers = 24M ordinary-week viewers x 90% core (C, illustrative) x 20% reached (A) x 20% active ATP pass buyers (A) = 864k. The control upgrade rate is 10% (A). An incremental upgrade is worth the season contribution less the passes that buyer would have bought anyway, in year one and again at 85% renewal in year two. The credit is paid to every treated upgrader, including the 10% who would have upgraded anyway.

In [ ]:
uc=pd.DataFrame([dict(item=k,value=v) for k,v in res['upgrade_cohort'].items()]);display(table(uc,'08_upgrade_cohort'))
grid=pd.DataFrame(res['pnl_grid']);display(table(grid,'08_pnl_grid'))
scen=pd.DataFrame([dict(scenario=k,**v) for k,v in res['scenarios'].items()]);display(table(scen,'08_pnl_scenarios'))
ref=res['reference_case']
steps=[('Acquired payers\nyear 1',ref['acquisition_y1']),('Upgrades, net\nof credit, year 1',ref['upgrade_y1']),('Committed spend\nincl. retention',-ref['spend']),('Year-one\nnet',None),('Year-two\ncontribution',ref['y2']),('Net at\n24 months',None)]
fig_,ax=plt.subplots(figsize=(10,4.6));run=0;tops=[0]
for i,(label,val) in enumerate(steps):
 if val is None:
  ax.bar(i,run/1e7,color='#193047',width=.6);ax.annotate(f'{run/1e7:+.2f}',(i,run/1e7),xytext=(0,4 if run>=0 else -12),textcoords='offset points',ha='center',fontsize=9);tops.append(run);continue
 ax.bar(i,val/1e7,bottom=run/1e7,color=COLORS[1] if val>=0 else COLORS[3],width=.6);end=run+val
 ax.annotate(f'{val/1e7:+.2f}',(i,max(run,end)/1e7),xytext=(0,4),textcoords='offset points',ha='center',fontsize=9);tops+= [run,end];run=end
ax.set_ylim(min(tops)/1e7-.35,max(tops)/1e7+.35)
ax.axhline(0,color='black',lw=.8);ax.set_xticks(range(len(steps)));ax.set_xticklabels([s[0] for s in steps],fontsize=8.5);ax.set_ylabel('INR crore')
ax.set_title(f"Reference case: 16% upgrade rate, 87.5% incrementality. Payback {ref['payback_months']:.1f} months",fontsize=12)
fig('08_reference_waterfall','Reference scenario only. It clears 24-month break-even but not the 15-month payback gate. Rights fee excluded.')

## 5. What the upgrade rate has to be
Two thresholds matter. **24-month break-even** is where the campaign repays its committed spend. **15-month payback** is the Gate 3 rule. Both are treatment upgrade rates against a 10% control.

In [ ]:
be=[]
for q in m.v('acquired_payers','incrementality_scenarios'):
 be.append(dict(incrementality=q,net_without_upgrades=res['without_upgrades'][str(q)],break_even_rate=res['break_even_treatment'][str(q)],rate_for_15m_payback=res['treatment_for_15m_payback'][str(q)],break_even_if_reserve_deployed=res['break_even_treatment_reserve_deployed'][str(q)]))
be=pd.DataFrame(be);display(table(be,'08_break_even_rates'))
sens=pd.DataFrame([dict(active_pass_buyer_share=float(s),incrementality=float(q),break_even_rate=r) for s,d in res['break_even_treatment_by_pass_buyer_share'].items() for q,r in d.items()]);display(table(sens,'08_break_even_by_pass_buyer_share'))
ren=pd.DataFrame([dict(season_renewal=float(k),break_even_rate_at_87_5=v) for k,v in res['break_even_treatment_by_renewal'].items()]);display(table(ren,'08_break_even_by_renewal'))
fig_,ax=plt.subplots(figsize=(10,4.2));rates=np.linspace(.10,.24,57)
for q,c in zip([1.0,.875,.6],[COLORS[1],COLORS[0],COLORS[3]]):ax.plot(rates*100,[m.pnl(t,q)['net_24m']/1e7 for t in rates],color=c,label=f'{q:.1%} of acquired payers incremental')
ax.axhline(0,color='black',lw=.8);ax.axvline(10,color='grey',ls=':');ax.text(10.2,ax.get_ylim()[1]*.85,'control 10%',fontsize=8,color='grey')
ax.set_xlabel('Treatment upgrade rate among eligible pass buyers (%)');ax.set_ylabel('Net at 24 months, INR crore');ax.legend(fontsize=8)
b_,p_=res['break_even_treatment'],res['treatment_for_15m_payback']
ax.set_title(f"Break-even at {b_['1.0']:.1%}-{b_['0.6']:.1%}; 15-month payback only at {p_['1.0']:.1%}-{p_['0.6']:.1%}")
fig('08_break_even_curve','Base assumptions: 20% of reached core are active pass buyers, 85% renewal, INR 36.66 credit cost on every treated upgrader. Performance reserve unspent.')

## 6. Sizing the gates to the economic test, not to mere detection
Gate 1 proves owned acquisition is incremental: the 95% lower bound on the lift must clear the lift at which incremental CAC equals INR 200. Gate 2 proves the upgrade engine: the 95% lower bound of (treatment − control) must clear the threshold minus the control rate. Both use 80% power. A true rate close to the threshold cannot be proved at any sensible cost; then the decision is continue or reallocate, never scale.

In [ ]:
g1=pd.DataFrame([res['gate1_owned_pilot']]);display(table(g1,'08_gate1_owned_pilot'))
g2=pd.DataFrame(res['gate2_upgrade_test']);display(table(g2,'08_gate2_upgrade_test'))

## 7. Independent checks
Key outputs are recomputed with Decimal arithmetic outside the model code.

In [ ]:
from decimal import Decimal as D
pass_c=D(89)/D('1.18')-D(89)*D('0.02')*D('1.18')-D(30)
season_c=D(399)/D('1.18')-D(399)*D('0.02')*D('1.18')-D(30)
credit=season_c-(D('354.5')/D('1.18')-D('354.5')*D('0.02')*D('1.18')-D(30))
owned=D('0.8631')/D('0.01')*D('1.15')
payers=D(9200000)/owned+D(4300000)/D(150)+D(2500000)/((D(50000)+D(12000))/D(390))+D(1200000)/D(180)+D(2500000)/(D(10)/D('0.03'))
spend=D(19700000)+D(3000000)+D(4320000)*D('0.8631')*D('0.5')*D('1.15')
acq24=payers*D('0.875')*D('1.625')*pass_c*D('1.4')
inc=D('0.06')*D(864000);upg=inc*(season_c-D('1.625')*pass_c)*(1+D('0.85'))-D('0.16')*D(864000)*credit
ref_net=acq24+upg-spend
checks={'pass_contribution':abs(float(pass_c)-res['unit_economics']['pass_contribution'])<1e-9,
 'season_contribution':abs(float(season_c)-res['unit_economics']['season_contribution'])<1e-9,
 'credit_cost':abs(float(credit)-res['unit_economics']['credit_cost_per_upgrade'])<1e-9,
 'attributed_payers':abs(float(payers)-res['envelope']['attributed_payers'])<1e-6,
 'reference_net_24m':abs(float(ref_net)-res['reference_case']['net_24m'])<1e-3,
 'break_even_is_zero':abs(m.pnl(res['break_even_treatment']['0.875'],0.875)['net_24m'])<1e-3,
 'payback_threshold_is_15':abs(m.pnl(res['treatment_for_15m_payback']['0.875'],0.875)['payback_months']-15)<1e-6,
 'envelope_reconciles':abs(sum(g['total'] for g in res['gate_allocation'])-30400000)<1e-6,
 'gates_reconcile':abs(gates.gate1.sum()-1e5)<1e-6 and abs(gates.gate2.sum()-75e5)<1e-6 and abs(gates.gate3.sum()-228e5)<1e-6,
 'only_owned_native_contests_clear_200_at_87_5':set(ch[ch.clears_200_at_87_5].channel)=={'owned_lifecycle','native_personalities','contests'},
 'upgrades_exclude_control':res['reference_case']['incremental_upgrades']==0.06*864000}
check('08_economics_model',checks)
r=res;ref=r['reference_case']
report('08_economics_findings',f"Committed acquisition of INR {r['envelope']['committed_acquisition']/1e7:.2f} Cr buys {r['envelope']['attributed_payers']/1e3:.1f}k attributed payers at INR {r['envelope']['attributed_blended_cac_incl_brand']:.0f} blended attributed CAC including brand and measurement, which meets INR 200 incremental CAC only at {r['envelope']['incrementality_needed_for_200']:.0%} incrementality or better. Each acquired payer contributes INR {r['unit_economics']['contribution_per_acquired_payer_24m']:.2f} over 24 months, so no channel repays its CAC on pass purchases alone; owned is closest. Without upgrades the campaign loses INR {-r['without_upgrades']['1.0']/1e7:.2f} to {-r['without_upgrades']['0.6']/1e7:.2f} Cr over 24 months. Against a 10% control upgrade rate among 864k eligible pass buyers, 24-month break-even needs a treatment upgrade rate of {r['break_even_treatment']['1.0']:.1%} to {r['break_even_treatment']['0.6']:.1%}, and 15-month payback needs {r['treatment_for_15m_payback']['1.0']:.1%} to {r['treatment_for_15m_payback']['0.6']:.1%}. The reference case (16%, 87.5%) nets INR {ref['net_24m']/1e7:.2f} Cr at 24 months with {ref['payback_months']:.1f}-month payback, so it would not pass Gate 3. Scale therefore waits for Gate 2 evidence. The performance reserve of INR {r['envelope']['performance_reserve']/1e7:.2f} Cr is released only on measured marginal incremental CAC at or below INR 200.")